# ChordSense Pro — Training on Colab
GPU: L4 24GB | Runtime: GPU

**v3 dataset** (with `bass_id`) phải được upload lên Drive tại:
`MyDrive/ChordSensePro/DE/data/export/v3/`

## 0. Config

In [ ]:
import os

# T4  16GB  → batch=16
# L4  24GB  → batch=32
# A100 40GB → batch=64
os.environ["NUM_WORKERS"] = "4"
os.environ["FP16"]        = "true"
os.environ["SEED"]        = "42"
print("Config set.")

## 1. Mount Drive + Clone Repo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!nvidia-smi

In [ ]:
DRIVE_PROJECT = "/content/drive/MyDrive/ChordSensePro"
REPO_DIR      = "/content/ChordSensePro"

if not os.path.exists(REPO_DIR):
    !git clone --branch qkhenh https://github.com/qkhenh/ChordSensePro.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin qkhenh

os.chdir(os.path.join(REPO_DIR, "AI"))
!pwd

## 2. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Prepare Data v3

Copy v3 từ Drive về /content (tránh I/O chậm khi đọc trực tiếp từ Drive khi train).

In [ ]:
DATA_V2_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v2")
DATA_V3_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v3")
DATA_V2 = "/content/data_v2"
DATA_DST = "/content/data_v3"

def _audio_ok(d):
    p = os.path.join(d, "train", "audio")
    return os.path.isdir(p) and len(os.listdir(p)) > 0

if _audio_ok(DATA_DST):
    print("v3 audio already at", DATA_DST)
elif _audio_ok(DATA_V3_SRC):
    print("Copying v3 from Drive...")
    !cp -r {DATA_V3_SRC} {DATA_DST}
else:
    print("v3 audio missing on Drive — rebuilding v3 from v2")
    if not os.path.exists(DATA_V2):
        print("Copying v2 from Drive (may take a few minutes)...")
        !cp -r {DATA_V2_SRC} {DATA_V2}
    else:
        print("v2 already at", DATA_V2)
    print("Building v3 locally...")
    !python ../DE/scripts/export_dataset_v3.py --v2-dir {DATA_V2} --out {DATA_DST}

import pandas as pd
df = pd.read_parquet(os.path.join(DATA_DST, "train", "chords.parquet"))
print("train rows:", len(df), "| bass_id:", "bass_id" in df.columns)
n_audio = len(os.listdir(os.path.join(DATA_DST, "train", "audio")))
print("train audio files:", n_audio)

---
## Phase 1 — Train Head Only
Chọn `RUN_NAME`: `v3_5s_p1` hoặc `v3_3s_p1`

In [ ]:
import yaml
from pathlib import Path

RUN_NAME = "v3_5s_p1"  # hoặc "v3_3s_p1"

cfg = yaml.safe_load(Path(f"configs/{RUN_NAME}.yaml").read_text())
cfg["data_dir"]       = DATA_DST
cfg["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", RUN_NAME)
cfg["results_csv"]    = os.path.join(DRIVE_PROJECT, "AI", "results", RUN_NAME, "results.csv")

os.makedirs(cfg["checkpoint_dir"], exist_ok=True)

colab_cfg = f"configs/colab_{RUN_NAME}.yaml"
Path(colab_cfg).write_text(yaml.dump(cfg, default_flow_style=False))
print("Config:", colab_cfg)
print("  data_dir      :", cfg["data_dir"])
print("  checkpoint_dir:", cfg["checkpoint_dir"])
print("  batch_size    :", cfg["train"]["batch_size"])
print("  epochs        :", cfg["train"]["epochs"])

In [ ]:
!python -m src.train --config {colab_cfg}

In [ ]:
# Resume Phase 1 nếu bị disconnect
P1_LAST = os.path.join(cfg["checkpoint_dir"], "last.pt")
if os.path.exists(P1_LAST):
    !python -m src.train --config {colab_cfg} --resume {P1_LAST}
else:
    print("No checkpoint found.")

---
## Phase 2 — LoRA Fine-tune MERT
Chạy sau khi Phase 1 xong. Cần `best.pt` từ Phase 1.

In [ ]:
P2_RUN_NAME = "v3_5s_p2"  # hoặc "v3_3s_p2"
P1_RUN_NAME = "v3_5s_p1"  # phải match với Phase 1 đã chạy

cfg_p2 = yaml.safe_load(Path(f"configs/{P2_RUN_NAME}.yaml").read_text())
cfg_p2["data_dir"]       = DATA_DST
cfg_p2["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", P2_RUN_NAME)
cfg_p2["results_csv"]    = os.path.join(DRIVE_PROJECT, "AI", "results", P2_RUN_NAME, "results.csv")

os.makedirs(cfg_p2["checkpoint_dir"], exist_ok=True)

colab_p2_cfg = f"configs/colab_{P2_RUN_NAME}.yaml"
Path(colab_p2_cfg).write_text(yaml.dump(cfg_p2, default_flow_style=False))

P1_BEST = os.path.join(DRIVE_PROJECT, "AI", "results", P1_RUN_NAME, "best.pt")
print("Phase 1 best.pt:", P1_BEST)
print("Exists:", os.path.exists(P1_BEST))

In [ ]:
!python -m src.train --config {colab_p2_cfg} --phase1_ckpt {P1_BEST}

In [ ]:
# Resume Phase 2 nếu bị disconnect
P2_LAST = os.path.join(cfg_p2["checkpoint_dir"], "last.pt")
if os.path.exists(P2_LAST):
    !python -m src.train --config {colab_p2_cfg} --resume {P2_LAST}
else:
    print("No Phase 2 checkpoint to resume.")

---
## Export ONNX

In [ ]:
P2_BEST  = os.path.join(cfg_p2["checkpoint_dir"], "best.pt")
OUT_ONNX = os.path.join(DRIVE_PROJECT, "AI", "results", f"model_{P2_RUN_NAME}.onnx")
!python -m src.export_onnx --config {colab_p2_cfg} --checkpoint {P2_BEST} --output {OUT_ONNX} --benchmark

---
## Quick Inference Test

In [ ]:
test_audio_dir = os.path.join(DATA_DST, "test", "audio")
test_files = sorted(os.listdir(test_audio_dir))[:3]
P2_BEST = os.path.join(cfg_p2["checkpoint_dir"], "best.pt")

for f in test_files:
    audio_path = os.path.join(test_audio_dir, f)
    !python -m src.infer --config {colab_p2_cfg} --checkpoint {P2_BEST} --audio {audio_path}